# Bluesky Data Scraping Test

In [1]:
# Load CSV file

import pandas as pd

bsky_sample_pd = pd.read_csv('Bluesky_SampleData.csv')

bsky_sample_pd.head()


,author.created_at,cid,record.media.merge,uri,indexed_at,record.created_at,record.text,record.langs,like_count,quote_count,...,record.embed.video.ref.link,record.reply.parent.uri,record.reply.parent.cid,record.reply.root.uri,record.reply.root.cid,record.hashtag,record.text.clean,record.text.clean.count,has.embed.external.title,record.media.merge.clean
0,2024-11-10T01:37:56.281Z,bafyreif3tsjqugonqeqh5rok5q73tptjkytjm6nvlspx6...,"Zelensky will be remembered for his true grit,...",at://did:plc:ujiuwn3nidrkc6u4esyjgqgo/app.bsky...,2025-03-03T04:59:06.949Z,2025-03-03T04:59:03.872Z,"Zelensky will be remembered for his true grit,...",['en'],3,0,...,NaN,NaN,NaN,NaN,NaN,[],"Zelensky will be remembered for his true grit,...",16,0,"Zelensky will be remembered for his true grit,..."
1,2024-12-02T00:10:05.047Z,bafyreifewhwblghlpfwworbkxmnqjyscof3ypvwsbfnm3...,youtube.com/watch?v=0n6l...\n\nAdam Mockler wi...,at://did:plc:hkrqzginr44yb4gghwi7yben/app.bsky...,2026-01-28T20:45:29.953Z,2026-01-28T20:45:27.912Z,youtube.com/watch?v=0n6l...\n\nAdam Mockler wi...,['en'],1,0,...,NaN,NaN,NaN,NaN,NaN,[],youtube.com/watch?v=0n6l... Adam Mockler with ...,27,1,youtube.com/watch?v=0n6l... Adam Mockler with ...
2,2023-11-12T03:12:51.798Z,bafyreidimj4jghhmhto6xtvxxteouvrvqllkrrgqciy2q...,Your post is tricky written. The headline is b...,at://did:plc:bxzxbo7az6q3vglzpnso5iu5/app.bsky...,2025-01-20T20:39:35.047Z,2025-01-20T20:39:34.296Z,Your post is tricky written. The headline is b...,['en'],0,0,...,NaN,at://did:plc:k5nskatzhyxersjilvtnz4lh/app.bsky...,bafyreibrc4g7boakhiusv3fffprxe2dnac5k2th654slk...,at://did:plc:k5nskatzhyxersjilvtnz4lh/app.bsky...,bafyreibrc4g7boakhiusv3fffprxe2dnac5k2th654slk...,[],Your post is tricky written. The headline is b...,45,0,Your post is tricky written. The headline is b...
3,2024-11-10T22:15:20.282Z,bafyreidmf63d52mh4y76ufbi4retkfbtkcdfhp57cpbob...,You’re right. But you had to know that any vot...,at://did:plc:m7oa6oudi5gw7gthner2xsoj/app.bsky...,2025-01-28T16:14:01.459Z,2025-01-28T16:14:00.885Z,You’re right. But you had to know that any vot...,['en'],0,0,...,NaN,at://did:plc:zodechfo7nrxqsam6yt5uhgi/app.bsky...,bafyreift4honmlmsqgpfpohmqkoo2u6g63oltwei6keoq...,at://did:plc:ydtsvzzsl6nlfkmnuooeqcmc/app.bsky...,bafyreibdvxsndnuexhyev4oodnl7f4exhayphewm5pdxe...,[],You’re right. But you had to know that any vot...,27,0,You’re right. But you had to know that any vot...
4,2024-11-12T16:01:11.511Z,bafyreih7pfrn3i67tsoqvrtceghrczojzhlib5ww2y5zk...,You want to talk about freedom Maga? \nThis ai...,at://did:plc:geyco4t7yp3ta7gh7qhrqsmh/app.bsky...,2025-05-05T17:32:32.484Z,2025-05-05T17:32:30.490Z,You want to talk about freedom Maga? \nThis ai...,['en'],1,0,...,bafkreigxpv562xrwsyv5bw634tfzyj4fde2vetjerngi2...,NaN,NaN,NaN,NaN,[],You want to talk about freedom Maga? This ain'...,11,0,You want to talk about freedom Maga? This ain'...


In [4]:
# Get URIs from the dataframe

URIs = bsky_sample_pd['uri'].tolist()

print(URIs[:5])  # Print the first 5 URIs


['at://did:plc:ujiuwn3nidrkc6u4esyjgqgo/app.bsky.feed.post/3ljh63ykafk2x', 'at://did:plc:hkrqzginr44yb4gghwi7yben/app.bsky.feed.post/3mdj5czlnms2l', 'at://did:plc:bxzxbo7az6q3vglzpnso5iu5/app.bsky.feed.post/3lg7733gfis2h', 'at://did:plc:m7oa6oudi5gw7gthner2xsoj/app.bsky.feed.post/3lgstxltcsc2t', 'at://did:plc:geyco4t7yp3ta7gh7qhrqsmh/app.bsky.feed.post/3logvlaxrhc2m']


In [7]:
import requests
import time

API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getPosts"
BATCH_SIZE = 25

def fetch_hydrated_posts(uris, batch_size=BATCH_SIZE, pause=1):
    """
    Fetch hydrated posts from the Bluesky API in batches.
    """

    hydrated_posts = []
    for i in range(0, len(uris), batch_size):
        batch_uris = uris[i:i + batch_size]
        response = requests.get(API_URL, params={"uris": batch_uris})
        if response.status_code == 200:
            hydrated_posts.extend(response.json().get('posts', []))
            time.sleep(pause)
        else:
            print(f"Error fetching posts for URIs {batch_uris}: {response.status_code}")
        time.sleep(1)  # To avoid hitting rate limits
    return hydrated_posts


if __name__ == "__main__":
    hydrated_posts = fetch_hydrated_posts(URIs)
    print(f"Fetched {len(hydrated_posts)} hydrated posts.")

Fetched 19 hydrated posts.


In [ ]:
for post in hydrated_posts:
    print(post["uri"], post.get("repostCount", 0), post.get("likeCount", 0), post.get("replyCount", 0), post.get("quoteCount", 0))

at://did:plc:ujiuwn3nidrkc6u4esyjgqgo/app.bsky.feed.post/3ljh63ykafk2x 1 3 1 0
at://did:plc:hkrqzginr44yb4gghwi7yben/app.bsky.feed.post/3mdj5czlnms2l 0 1 0 0
at://did:plc:bxzxbo7az6q3vglzpnso5iu5/app.bsky.feed.post/3lg7733gfis2h 0 0 0 0
at://did:plc:m7oa6oudi5gw7gthner2xsoj/app.bsky.feed.post/3lgstxltcsc2t 0 0 1 0
at://did:plc:geyco4t7yp3ta7gh7qhrqsmh/app.bsky.feed.post/3logvlaxrhc2m 0 1 1 0
at://did:plc:fyfguitrzwh6z2njvmyldnvj/app.bsky.feed.post/3lkcon6bj6k26 0 2 0 0
at://did:plc:kxl3oedc3kcunavmmkqw5wrv/app.bsky.feed.post/3lbfweuefhs27 0 0 0 0
at://did:plc:lrix7yjvlvg6yb7t7rp55mjv/app.bsky.feed.post/3m7j5ipdl422i 0 0 0 0
at://did:plc:a6l33kt5nxqcwgumrxhrgcss/app.bsky.feed.post/3lbhsj5mpq22z 0 0 0 0
at://did:plc:g4yay2rbkl2ozds46rt2qwxr/app.bsky.feed.post/3mdkl653ym22m 0 0 0 0
at://did:plc:s6thjs3jzkt2axf5u5dzpxob/app.bsky.feed.post/3l5vih6cxzb2f 0 1 2 0
at://did:plc:izngy4b2wl4ukbxowee7facy/app.bsky.feed.post/3lf7zo5j2hc2t 0 0 0 0
at://did:plc:jmvixb63l2aixziy6oybtawo/app.bsky.feed.

In [16]:
import json

print(json.dumps(hydrated_posts[0], indent=2))

{
  "uri": "at://did:plc:ujiuwn3nidrkc6u4esyjgqgo/app.bsky.feed.post/3ljh63ykafk2x",
  "cid": "bafyreif3tsjqugonqeqh5rok5q73tptjkytjm6nvlspx6ejwkvvucawjee",
  "author": {
    "did": "did:plc:ujiuwn3nidrkc6u4esyjgqgo",
    "handle": "claude-oneshot.bsky.social",
    "avatar": "https://cdn.bsky.app/img/avatar/plain/did:plc:ujiuwn3nidrkc6u4esyjgqgo/bafkreifuhufluuo6trxddsgxdmbzci4embebwjlrulncyht3xxdhlpbgcy",
    "associated": {
      "activitySubscription": {
        "allowSubscriptions": "followers"
      }
    },
    "labels": [],
    "createdAt": "2024-11-10T01:37:56.281Z"
  },
  "record": {
    "$type": "app.bsky.feed.post",
    "createdAt": "2025-03-03T04:59:03.872Z",
    "embed": {
      "$type": "app.bsky.embed.record",
      "record": {
        "cid": "bafyreigbag327uuafzatfbpnmsxthlgrdwcfrmtp6htfbuxdktiiy5mzpu",
        "uri": "at://did:plc:ioblxbjv36hzkeequ6q3ttb6/app.bsky.feed.post/3ljgjxf4wrc2u"
      }
    },
    "langs": [
      "en"
    ],
    "text": "Zelensky will be rem